# VERA — CALVIN Ablation Training
**9 ablations × 3 seeds on CALVIN D→D**

### Before running:
1. `Runtime → Change runtime type → T4 GPU` (free) or A100 (Pro)
2. Run cells top to bottom — dataset downloads automatically inside Colab (first time only, saves to Drive)

### If session dies mid-run:
- Checkpoints are saved to Drive after every epoch — nothing is lost
- Re-run all setup cells (dataset skips re-download if already on Drive)
- In Cell 7, set `START_FROM` to the ablation index that was running when it died

In [1]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:  ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → GPU')

GPU available: True
Device: Tesla T4
VRAM:   15.6 GB


In [2]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints and results will be saved here — survives session crashes
DRIVE_ROOT = '/content/drive/MyDrive/VERA_CALVIN'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted. Output dir:', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted. Output dir: /content/drive/MyDrive/VERA_CALVIN


In [3]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# torch / numpy / PIL already in Colab — only need CLIP and ftfy
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q ftfy regex

import clip, numpy, PIL, yaml
print('clip:', clip.__version__ if hasattr(clip,'__version__') else 'ok')
print('torch:', torch.__version__)
print('All dependencies ready')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
clip: ok
torch: 2.10.0+cu128
All dependencies ready


In [4]:
# ── Cell 4: Clone repo ───────────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR  = '/content/RLConditionedVLA'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!ls

Cloning into '/content/RLConditionedVLA'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 175 (delta 12), reused 8 (delta 2), pack-reused 147 (from 1)
Receiving objects: 100% (175/175), 41.99 MiB | 22.27 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Working dir: /content/RLConditionedVLA
configs  envs	     README.md	     requirements.txt  VERA_CALVIN_Colab.ipynb
data	 evaluation  references.bib  scripts
docs	 models      REPORT.md	     training


In [6]:
# ── Optional: Clear Drive cache (run only if you want to re-download everything) ─
# Normally skip this cell.
# Run it only if the cache is corrupt or you want to force a fresh download.

CLEAR_CACHE = False   # ← change to True then run this cell to wipe cache

if CLEAR_CACHE:
    import shutil, os
    cache = f'{DRIVE_ROOT}/calvin_cache'
    cdir  = f'{DRIVE_ROOT}/calvin_cdir.json'
    if os.path.exists(cache):
        shutil.rmtree(cache)
        print(f'Removed {cache}')
    if os.path.exists(cdir):
        os.remove(cdir)
        print(f'Removed {cdir}')
    print('Cache cleared — re-run Cell 5 to re-download')
else:
    print('Cache intact (set CLEAR_CACHE = True to wipe)')


In [8]:
# ── Cell 5: CALVIN dataset — ZIP64 surgical range-request extraction ──────────
# Server does NOT serve individual files (404). The only URL is the 177 GB zip.
# We parse the ZIP64 central directory via HTTP range requests and download
# exactly the files we need (~3-8 GB total) without touching the rest.
#
# Drive cache layout:  $DRIVE_ROOT/calvin_cache/{training,validation}/
#   episode_XXXXXXX.npz  — cached NPZ frames (survive session crashes)
#   lang_annotations/auto_lang_ann.npy
# $DRIVE_ROOT/calvin_cdir.json  — parsed ZIP central directory (72 MB, built once)
#
import os, io, struct, json as _json, shutil, urllib.request, urllib.error
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

CALVIN_ZIP_URL = 'http://calvin.cs.uni-freiburg.de/dataset/task_D_D.zip'
CALVIN_PATH    = '/content/calvin_data/task_D_D'
DRIVE_CACHE    = f'{DRIVE_ROOT}/calvin_cache'
CDIR_CACHE     = f'{DRIVE_ROOT}/calvin_cdir.json'

TRAIN_EP = 1000   # annotated training episodes to use
VAL_EP   = 200    # annotated validation episodes to use

for split in ['training', 'validation']:
    os.makedirs(f'{CALVIN_PATH}/{split}/lang_annotations', exist_ok=True)
    os.makedirs(f'{DRIVE_CACHE}/{split}/lang_annotations', exist_ok=True)

# ── Byte-range HTTP helper ────────────────────────────────────────────────────
def range_get(url, start, end, timeout=90):
    # Download bytes [start, end] inclusive from url
    req = urllib.request.Request(url)
    req.add_header('Range', f'bytes={start}-{end}')
    resp = urllib.request.urlopen(req, timeout=timeout)
    chunks = []
    while True:
        chunk = resp.read(4 * 1024 * 1024)
        if not chunk:
            break
        chunks.append(chunk)
    return b''.join(chunks)

# ── ZIP64 central directory parser (built once, cached to Drive) ─────────────
_file_index = None   # {internal_name: [local_hdr_offset, compressed_size]}

def get_file_index():
    global _file_index
    if _file_index is not None:
        return _file_index

    if os.path.exists(CDIR_CACHE):
        print('Loading cached ZIP central directory from Drive...')
        with open(CDIR_CACHE) as f:
            _file_index = _json.load(f)
        print(f'  {len(_file_index):,} files indexed')
        return _file_index

    print('Parsing ZIP64 central directory (downloads ~73 MB once)...')

    # Step 1: Get total zip size
    resp = urllib.request.urlopen(
        urllib.request.Request(CALVIN_ZIP_URL, method='HEAD'), timeout=15)
    zip_size = int(resp.headers['Content-Length'])
    print(f'  ZIP size: {zip_size / 1e9:.2f} GB')

    # Step 2: Read last 64 KB to find ZIP64 EOCD locator (sig = PK\x06\x07)
    tail = range_get(CALVIN_ZIP_URL, zip_size - 65536, zip_size - 1)
    loc_sig = b'\x50\x4b\x06\x07'
    li = tail.rfind(loc_sig)
    assert li >= 0, 'ZIP64 EOCD locator not found in last 64 KB'
    eocd64_off = struct.unpack_from('<Q', tail, li + 8)[0]

    # Step 3: Read ZIP64 end-of-central-directory record (56 bytes)
    eocd64 = range_get(CALVIN_ZIP_URL, eocd64_off, eocd64_off + 55)
    assert eocd64[:4] == b'\x50\x4b\x06\x06', 'Bad ZIP64 EOCD signature'
    cdir_size   = struct.unpack_from('<Q', eocd64, 40)[0]
    cdir_offset = struct.unpack_from('<Q', eocd64, 48)[0]
    print(f'  Central dir offset: {cdir_offset:,}  size: {cdir_size / 1e6:.1f} MB')

    # Step 4: Download the entire central directory
    cdir = range_get(CALVIN_ZIP_URL, cdir_offset, cdir_offset + cdir_size - 1, timeout=180)
    print(f'  Downloaded {len(cdir) / 1e6:.1f} MB')

    # Step 5: Parse central directory entries
    _file_index = {}
    pos = 0
    cdir_sig = b'\x50\x4b\x01\x02'
    while pos + 46 <= len(cdir) and cdir[pos:pos+4] == cdir_sig:
        uncomp_sz = struct.unpack_from('<I', cdir, pos + 24)[0]
        comp_sz   = struct.unpack_from('<I', cdir, pos + 20)[0]
        name_len  = struct.unpack_from('<H', cdir, pos + 28)[0]
        extra_len = struct.unpack_from('<H', cdir, pos + 30)[0]
        cmnt_len  = struct.unpack_from('<H', cdir, pos + 32)[0]
        lhdr_off  = struct.unpack_from('<I', cdir, pos + 42)[0]

        name  = cdir[pos+46 : pos+46+name_len].decode('utf-8', errors='replace')
        extra = cdir[pos+46+name_len : pos+46+name_len+extra_len]

        # Parse ZIP64 extended information extra field (tag 0x0001)
        ep = 0
        while ep + 4 <= len(extra):
            ex_id = struct.unpack_from('<H', extra, ep)[0]
            ex_sz = struct.unpack_from('<H', extra, ep + 2)[0]
            if ex_id == 0x0001:
                vp = ep + 4
                # Fields appear in order: uncomp(8)? comp(8)? offset(8)?
                # — only present when the 4-byte field is 0xFFFFFFFF
                if uncomp_sz == 0xFFFFFFFF and vp + 8 <= ep + 4 + ex_sz:
                    uncomp_sz = struct.unpack_from('<Q', extra, vp)[0]; vp += 8
                if comp_sz == 0xFFFFFFFF and vp + 8 <= ep + 4 + ex_sz:
                    comp_sz   = struct.unpack_from('<Q', extra, vp)[0]; vp += 8
                if lhdr_off == 0xFFFFFFFF and vp + 8 <= ep + 4 + ex_sz:
                    lhdr_off  = struct.unpack_from('<Q', extra, vp)[0]
                break
            ep += 4 + ex_sz

        # Strip internal zip prefix
        if name.startswith('task_D_D/'):
            name = name[len('task_D_D/'):]

        if name and not name.endswith('/'):
            _file_index[name] = [lhdr_off, comp_sz]

        pos += 46 + name_len + extra_len + cmnt_len

    print(f'  Parsed {len(_file_index):,} entries')
    with open(CDIR_CACHE, 'w') as f:
        _json.dump(_file_index, f)
    print(f'  Saved index to Drive: {CDIR_CACHE}')
    return _file_index

# ── Download a single file from ZIP via range request ─────────────────────────
def download_from_zip(internal_name, timeout=90):
    # Extract one file from the remote ZIP using byte-range requests
    idx = get_file_index()
    key = internal_name.lstrip('/')
    if key not in idx:
        raise KeyError(f'Not in index: {key}')
    lhdr_off, comp_sz = idx[key]

    # Read local file header to find data start offset
    lhdr = range_get(CALVIN_ZIP_URL, lhdr_off, lhdr_off + 29, timeout=15)
    assert lhdr[:4] == b'\x50\x4b\x03\x04', f'Bad local header at {lhdr_off}'
    method    = struct.unpack_from('<H', lhdr,  8)[0]
    name_len  = struct.unpack_from('<H', lhdr, 26)[0]
    extra_len = struct.unpack_from('<H', lhdr, 28)[0]
    data_off  = lhdr_off + 30 + name_len + extra_len

    # Download compressed payload
    raw = range_get(CALVIN_ZIP_URL, data_off, data_off + comp_sz - 1, timeout=timeout)

    if method == 8:       # DEFLATE
        import zlib
        return zlib.decompress(raw, -15)
    elif method == 0:     # Stored (no compression)
        return raw
    else:
        raise ValueError(f'Unsupported zip method {method}')

# ── Get lang_annotations for a split ──────────────────────────────────────────
def ensure_lang_ann(split):
    local_path = f'{CALVIN_PATH}/{split}/lang_annotations/auto_lang_ann.npy'
    cache_path = f'{DRIVE_CACHE}/{split}/lang_annotations/auto_lang_ann.npy'

    if os.path.exists(local_path):
        pass   # already on local disk
    elif os.path.exists(cache_path):
        shutil.copy2(cache_path, local_path)
        print(f'  [{split}] lang_annotations: loaded from Drive cache')
    else:
        print(f'  [{split}] lang_annotations: downloading via range request...')
        data = download_from_zip(f'{split}/lang_annotations/auto_lang_ann.npy')
        with open(local_path, 'wb') as f:
            f.write(data)
        shutil.copy2(local_path, cache_path)
        print(f'  [{split}] lang_annotations: {len(data)/1e6:.2f} MB saved')

    ann = np.load(local_path, allow_pickle=True).item()
    return ann['info']['indx'], ann['language']['task']

# ── Get episode NPZs for annotated episodes ───────────────────────────────────
def ensure_episodes(split, indx, max_ep):
    local_dir = f'{CALVIN_PATH}/{split}'
    cache_dir = f'{DRIVE_CACHE}/{split}'

    needed = sorted({i for s, e in indx[:max_ep] for i in range(int(s), int(e)+1)})
    missing     = [i for i in needed if not os.path.exists(f'{local_dir}/episode_{i:07d}.npz')]
    in_cache    = [i for i in missing if  os.path.exists(f'{cache_dir}/episode_{i:07d}.npz')]
    to_download = [i for i in missing if not os.path.exists(f'{cache_dir}/episode_{i:07d}.npz')]

    print(f'\n[{split}] {len(needed)} frames needed for {min(max_ep, len(indx))} episodes')
    print(f'  On local disk: {len(needed) - len(missing):,}')
    print(f'  In Drive cache: {len(in_cache):,}')
    print(f'  To download:   {len(to_download):,}')

    # Copy from Drive cache
    for i in tqdm(in_cache, desc=f'{split}: copy from cache', leave=False):
        shutil.copy2(f'{cache_dir}/episode_{i:07d}.npz',
                     f'{local_dir}/episode_{i:07d}.npz')

    # Download missing via range requests
    if to_download:
        print(f'  Downloading {len(to_download):,} files via range requests '
              f'(may take a while)...')
        failed = 0
        for i in tqdm(to_download, desc=f'{split}: download'):
            fname  = f'episode_{i:07d}.npz'
            lpath  = f'{local_dir}/{fname}'
            cpath  = f'{cache_dir}/{fname}'
            try:
                data = download_from_zip(f'{split}/{fname}')
                with open(lpath, 'wb') as f:
                    f.write(data)
                shutil.copy2(lpath, cpath)
            except Exception as exc:
                failed += 1
                if failed <= 5:
                    print(f'  ! Failed {fname}: {exc}')
        if failed:
            print(f'  WARNING: {failed} files failed — will be skipped by loader')

    on_disk = list(Path(local_dir).glob('episode_*.npz'))
    print(f'  Total on local disk: {len(on_disk):,}')

# ── Main ──────────────────────────────────────────────────────────────────────
print('=== Step 1: lang_annotations ===')
train_indx, train_tasks = ensure_lang_ann('training')
val_indx,   val_tasks   = ensure_lang_ann('validation')
print(f'  training:   {len(train_indx):,} annotated episodes')
print(f'  validation: {len(val_indx):,} annotated episodes')

print('\n=== Step 2: episode NPZs ===')
ensure_episodes('training',   train_indx, TRAIN_EP)
ensure_episodes('validation', val_indx,   VAL_EP)

# ── Verify ────────────────────────────────────────────────────────────────────
train_npz = list(Path(f'{CALVIN_PATH}/training').glob('episode_*.npz'))
val_npz   = list(Path(f'{CALVIN_PATH}/validation').glob('episode_*.npz'))
total_gb  = sum(f.stat().st_size for f in train_npz + val_npz) / 1e9
print(f'\nLocal disk — training: {len(train_npz):,}  validation: {len(val_npz):,}  ({total_gb:.2f} GB)')
assert len(train_npz) > 50, f'Too few training NPZs ({len(train_npz)}) — check errors above'
print('CALVIN_PATH:', CALVIN_PATH)
print('\nDone ✓')


In [ ]:
# ── Cell 6: Smoke test — verify loader works before full run ─────────────────
import sys
sys.path.insert(0, REPO_DIR)

from data.trajectory_dataset import load_calvin

print('Loading a few CALVIN episodes to verify loader...')
eps = load_calvin(CALVIN_PATH, split='training')

print(f'Episodes loaded: {len(eps)}')
print(f'Keys: {list(eps[0].keys())}')
print(f'Frames shape:  {eps[0]["frames"].shape}')
print(f'Actions shape: {eps[0]["actions"].shape}  dtype={eps[0]["actions"].dtype}')
print(f'Action range:  {eps[0]["actions"].min()} – {eps[0]["actions"].max()}  (expect 0–13)')
print(f'Action vectors:{eps[0]["action_vectors"].shape}  (expect T×7)')
print(f'Instruction:   {eps[0]["instruction"]}')
print('Loader OK')

In [ ]:
# ── Cell 7: Configure run ─────────────────────────────────────────────────────
# ↓ ONLY change these two values ↓

START_FROM = 0    # Resume from ablation index if session died (0 = start fresh)
                  # Indices: 0=full_vera  1=bc_baseline  2=no_exp  3=no_act
                  #          4=no_lang    5=no_history_tf 6=no_reward_gate
                  #          7=no_dual_head  8=corrupted_conseq

DRY_RUN = False   # Set True first to verify (2 epochs, synthetic data, ~2 min)
                  # Set False for the real run

# ─────────────────────────────────────────────────────────────────────────────
# Point checkpoints to Drive so they survive session crashes
import yaml, os

cfg_path = f'{REPO_DIR}/configs/calvin_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Redirect output to Drive
cfg['data']['episodes_path']   = CALVIN_PATH
cfg['training']['output_dir']  = f'{DRIVE_ROOT}/checkpoints'
cfg['training']['device']      = 'cuda'
cfg['training']['num_workers'] = 2    # Colab works better with 2

# Save patched config
colab_cfg_path = f'{REPO_DIR}/configs/calvin_colab_runtime.yaml'
with open(colab_cfg_path, 'w') as f:
    yaml.dump(cfg, f)

print('Config ready')
print(f'  Dataset:  {CALVIN_PATH}')
print(f'  Output:   {DRIVE_ROOT}/checkpoints')
print(f'  Epochs:   {cfg["training"]["epochs"]} (max, early stopping patience=10)')
print(f'  Ablations to run: {9 - START_FROM} (starting from index {START_FROM})')
print(f'  Seeds: 42, 123, 456')
print(f'  DRY RUN: {DRY_RUN}')

In [ ]:
# ── Cell 8: Run ablations ─────────────────────────────────────────────────────
# This cell runs all 9 ablations × 3 seeds sequentially.
# Checkpoints save to Drive after every epoch — safe to interrupt and resume.
# Expected time on T4:  ~20-40 min per (ablation × seed)  →  ~9-18 hours total
# Expected time on A100: ~8-15 min per (ablation × seed)  →  ~4-7  hours total

import subprocess, sys

cmd = [
    sys.executable,
    f'{REPO_DIR}/scripts/run_calvin_ablations.py',
    '--calvin_path', CALVIN_PATH,
    '--config',      colab_cfg_path,
    '--out',         f'{DRIVE_ROOT}/checkpoints',
    '--start_from',  str(START_FROM),
]
if DRY_RUN:
    cmd.append('--dry_run')

print('Running:', ' '.join(cmd))
print('='*65)

# Run with live output
result = subprocess.run(cmd, cwd=REPO_DIR)
print('='*65)
print('Exit code:', result.returncode)

In [ ]:
# ── Cell 9: View results ──────────────────────────────────────────────────────
import json, numpy as np

results_path = f'{DRIVE_ROOT}/checkpoints/calvin_ablation_summary.json'

with open(results_path) as f:
    results = json.load(f)

print(f'  {"Ablation":<46} {"Mean":>7}  {"Std":>6}  Seeds')
print(f'  {"-"*46} {"-"*7}  {"-"*6}  {"-"*22}')
for slug, v in results.items():
    seeds = '  '.join(f'{x:.4f}' for x in v['seeds'].values())
    print(f'  {v["display"]:<46} {v["mean_val_acc"]:.4f}  {v["std_val_acc"]:.4f}  {seeds}')

In [ ]:
# ── Cell 10: Download results JSON to local machine ───────────────────────────
from google.colab import files
import shutil, os

# Bundle the summary JSON + all training logs into one zip
zip_path = '/content/calvin_results.zip'
shutil.make_archive(
    '/content/calvin_results', 'zip',
    f'{DRIVE_ROOT}/checkpoints'
)
files.download(zip_path)
print('Download started')